# 02 · Historical SpaceX launch table — web scraping

**Project context and attribution.** Developed for the IBM Applied Data Science Capstone (Coursera / IBM Skills Network). IBM supplied the original educational project framework and public teaching datasets; this is a reorganized, independently worded notebook incorporating the student's analysis and subsequent code corrections, with AI-assisted review. The archived teaching data are historical and are **not** a current account of SpaceX launches.

**Run environment:** Python 3 in VS Code's Jupyter notebook extension. Install the project dependencies outside the notebook using the repository's `requirements.txt`. There are no `pip`, `piplite`, JavaScript `fetch`, or notebook-environment-specific installation cells here.

## Aim and scope

Parse the **archived June 2021 revision** of Wikipedia's Falcon launch history, rather than the continually changing current page. Extract launch rows with BeautifulSoup into a structured table. Keep *launch outcome* (mission) distinct from *booster landing* (first stage). The archived HTML structure is an external dependency.

In [1]:
from pathlib import Path
import re
import unicodedata
import pandas as pd
import requests
from bs4 import BeautifulSoup

OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)

SOURCE_URL = (
    "https://en.wikipedia.org/w/index.php?"
    "title=List_of_Falcon_9_and_Falcon_Heavy_launches&oldid=1027686922"
)
HEADERS = {"User-Agent": "Mozilla/5.0 (compatible; educational-data-science-project/1.0)"}

## Download and parse the archived HTML

If Wikipedia rate-limits a request, wait and retry. A manually saved copy named `spacex_wikipedia_2021.html` in the notebook directory can also be used to run offline.

In [2]:
saved_html = Path("spacex_wikipedia_2021.html")
if saved_html.exists():
    html_data = saved_html.read_text(encoding="utf-8")
    print("Using locally saved Wikipedia HTML snapshot.")
else:
    response = requests.get(SOURCE_URL, headers=HEADERS, timeout=40)
    response.raise_for_status()
    html_data = response.text
    print("Downloaded archived Wikipedia revision:", response.status_code)

soup = BeautifulSoup(html_data, "html.parser")
print("Document title:", soup.title.get_text(" ", strip=True) if soup.title else "N/A")

Downloaded archived Wikipedia revision: 200
Document title: List of Falcon 9 and Falcon Heavy launches - Wikipedia


## Parsing helpers

Normalize whitespace, tolerate cells without hyperlinks, and keep unknown payload masses as missing (`NaN`) rather than incorrectly assigning zero kilograms.

In [3]:
def clean_text(cell):
    return " ".join(cell.stripped_strings).strip() if cell is not None else ""


def date_time(cell):
    parts = [s.strip() for s in cell.stripped_strings if s.strip()]
    return (parts[0] if parts else "", parts[1] if len(parts) > 1 else "")


def get_mass(cell):
    text = unicodedata.normalize("NFKC", clean_text(cell))
    match = re.search(r"([\d,]+(?:\.\d+)?)\s*kg\b", text, flags=re.IGNORECASE)
    return float(match.group(1).replace(",", "")) if match else float("nan")

COLUMNS = [
    "Flight No.", "Date", "Time", "Version Booster", "Launch site",
    "Payload", "Payload mass", "Orbit", "Customer",
    "Launch outcome", "Booster landing"
]

## Identify launch tables and accumulate records

A row is included only if its header is a launch number and it has the expected nine data cells. **Append a complete row per launch** rather than overwriting dictionary entries. This fixes the original notebook's primary bug.

In [4]:
launch_tables = soup.select("table.wikitable.plainrowheaders.collapsible")
if not launch_tables:
    # Wikipedia sometimes changes wrapper classes. Only numeric-header rows
    # with the required number of data columns will be accepted below.
    launch_tables = soup.select("table.wikitable")
print("Candidate launch tables:", len(launch_tables))

records = []
seen_numbers = set()
for table in launch_tables:
    for tr in table.find_all("tr"):
        th = tr.find("th", recursive=False)
        if th is None:
            continue
        flight_text = re.sub(r"\[[^]]*]", "", clean_text(th)).strip()
        if not re.fullmatch(r"\d+", flight_text):
            continue
        cells = tr.find_all("td", recursive=False)
        if len(cells) < 9:
            continue
        flight_number = int(flight_text)
        if flight_number in seen_numbers:
            continue
        seen_numbers.add(flight_number)

        date, time = date_time(cells[0])
        records.append({
            "Flight No.": flight_number,
            "Date": date,
            "Time": time,
            "Version Booster": clean_text(cells[1]),
            "Launch site": clean_text(cells[2]),
            "Payload": clean_text(cells[3]),
            "Payload mass": get_mass(cells[4]),
            "Orbit": clean_text(cells[5]),
            "Customer": clean_text(cells[6]),
            "Launch outcome": clean_text(cells[7]),
            "Booster landing": clean_text(cells[8]),
        })

df = pd.DataFrame.from_records(records, columns=COLUMNS)
if df.empty:
    raise ValueError(
        "No launch rows were extracted. Wikipedia's archived table structure may "
        "have changed, the response may be an error page, or the HTML is incomplete."
    )
print("Scraped historical launch rows:", len(df))
df.head()

Candidate launch tables: 9
Scraped historical launch rows: 121


,Flight No.,Date,Time,Version Booster,Launch site,Payload,Payload mass,Orbit,Customer,Launch outcome,Booster landing
0,1,"4 June 2010,",18:45,F9 v1.0 [ 7 ] B0003.1 [ 8 ],"CCAFS , SLC-40",Dragon Spacecraft Qualification Unit,NaN,LEO,SpaceX,Success,Failure [ 9 ] [ 10 ] (parachute)
1,2,"8 December 2010,",15:43,F9 v1.0 [ 7 ] B0004.1 [ 8 ],"CCAFS , SLC-40",Dragon demo flight C1 (Dragon C101),NaN,LEO ( ISS ),NASA ( COTS ) NRO,Success [ 9 ],Failure [ 9 ] [ 14 ] (parachute)
2,3,"22 May 2012,",07:44,F9 v1.0 [ 7 ] B0005.1 [ 8 ],"CCAFS , SLC-40",Dragon demo flight C2+ [ 18 ] (Dragon C102),525.0,LEO ( ISS ),NASA ( COTS ),Success [ 20 ],No attempt
3,4,"8 October 2012,",00:35,F9 v1.0 [ 7 ] B0006.1 [ 8 ],"CCAFS , SLC-40",SpaceX CRS-1 [ 22 ] (Dragon C103),4700.0,LEO ( ISS ),NASA ( CRS ),Success,No attempt
4,5,"1 March 2013,",15:10,F9 v1.0 [ 7 ] B0007.1 [ 8 ],"CCAFS , SLC-40",SpaceX CRS-2 [ 22 ] (Dragon C104),4877.0,LEO ( ISS ),NASA ( CRS ),Success,No attempt


## Validate extraction and export

Check that all columns have the same number of rows, launch numbers are unique, and missing payload masses remain explicitly missing. If the result seems too small, visually inspect the downloaded archived page before citing any analysis.

In [5]:
assert df["Flight No."].is_unique
assert len(df) == len(records)
print("Missing payload masses:", df["Payload mass"].isna().sum())
print("Flight number range:", df["Flight No."].min(), "to", df["Flight No."].max())
print("Launch outcomes (raw text):")
print(df["Launch outcome"].value_counts().head(10))
if len(df) < 50:
    print("WARNING: Fewer than 50 launches parsed; verify the archived HTML manually.")

output_file = OUTPUT_DIR / "spacex_web_scraped.csv"
df.to_csv(output_file, index=False)
print("Saved:", output_file.resolve())
df.tail()

Missing payload masses: 5
Flight number range: 1 to 121
Launch outcomes (raw text):
Launch outcome
Success           88
Success [ 9 ]      1
Success [ 20 ]     1
Success [ 30 ]     1
Success [ 37 ]     1
Success [ 42 ]     1
Success [ 51 ]     1
Success [ 56 ]     1
Success [ 59 ]     1
Success [ 63 ]     1
Name: count, dtype: int64
Saved: C:\Users\alexa\AppData\Local\Temp\9af4be9a-ebe6-416d-a858-3688b340c92c_SpaceX_Capstone_Four_VSCode_Notebooks.zip.92c\outputs\spacex_web_scraped.csv


,Flight No.,Date,Time,Version Booster,Launch site,Payload,Payload mass,Orbit,Customer,Launch outcome,Booster landing
116,117,9 May 2021,06:42,F9 B5 ♺ B1051.10 [ 657 ],"CCSFS , SLC-40",Starlink 27 v1.0 (60 satellites),15600.0,LEO,SpaceX,Success,Success (drone ship)
117,118,15 May 2021,22:56,F9 B5 ♺ B1058.8 [ 660 ],"KSC , LC-39A [ 661 ]",Starlink 26 v1.0 (52 Satellites) Capella -6 & ...,14000.0,LEO,SpaceX Capella Space and Tyvak,Success,Success (drone ship)
118,119,26 May 2021,18:59,F9 B5 ♺ B1063.2 [ 665 ],"CCSFS , SLC-40 [ 665 ]",Starlink 28 v1.0 (60 Satellites) [ 665 ],15600.0,LEO,SpaceX,Success,Success (drone ship)
119,120,3 June 2021,17:29,F9 B5 B1067.1 [ 668 ],"KSC , LC-39A",SpaceX CRS-22 ( Dragon C209.1 ),3328.0,LEO ( ISS ),NASA ( CRS ),Success,Success (drone ship)
120,121,6 June 2021,04:26,F9 B5 ♺ B1061.3,"CCSFS , SLC-40",SXM-8 [ 674 ],7000.0,GTO,Sirius XM,Success,Success (drone ship)


## What this notebook demonstrates

- HTTP fetching of an archived public webpage
- HTML table parsing with BeautifulSoup
- Safe row accumulation and column validation
- Separation of launch/mission results from first-stage landing results

**Limitations:** HTML layouts can change; text fields and dates remain close to their original historical presentation. The numeric payload masses are parsed only where `kg` is present.

**Credit:** IBM Skills Network / Coursera Applied Data Science Capstone for the original historical web-scraping exercise and data source; implementation reorganized and corrected for VS Code.